# FER2013 Flat Array Model Training

## Emotion Face Classifier

This notebook performs data preprocessing and analysis with FER 2013 data only.

## NOTE: Only training metrics created here, test metrics are more meaningful

In [2]:
import os
import numpy as np
import pandas as pd
import importlib
import matplotlib.pyplot as plt

In [3]:
def check_directory_name(target_name) -> bool:
    """
    Check if the current directory name matches the target_name.
    If not, move up a directory and repeat the check.
    
    Args:
        target_name (str): The directory name to match.
        
    Returns:
        bool: True if the current directory name matches the target_name, False otherwise.
    """
    # Get the current directory path
    current_dir = os.getcwd()
    
    # Extract the directory name from the path
    current_dir_name = os.path.basename(current_dir)
    
    # Check if the current directory name matches the target_name
    if current_dir_name == target_name:
        print(f'Directory set to {current_dir}, matches target dir sting {target_name}.')
        return True
    else:
        # Move up a directory
        os.chdir('..')
        # Check if we have reached the root directory
        if os.getcwd() == current_dir:
            return False
        # Recursively call the function to check the parent directory
        return check_directory_name(target_name)

In [4]:
main_dir = 'EmotionFaceClassifier'
check_directory_name(main_dir)

Directory set to /Users/dsl/Documents/GitHub/EmotionFaceClassifier, matches target dir sting EmotionFaceClassifier.


True

In [5]:
from src.main import (
    convert_pixels_to_array,
    str_to_array,
    load_config,
    create_models,
    get_classification_metrics,
    save_model,
    to_json
)

## Data Import and Prep

In [7]:
df_path = 'data/fer2013/fer2013.csv'
df = pd.read_csv(df_path)

In [8]:
# Load common dicts from json config file
common_dicts = load_config('./configs/basics.json')
print(common_dicts.keys())

dict_keys(['usage_dict', 'emo_dict', 'emo_color_dict', 'output_col_order'])


In [9]:
# Load in key dicts from json for data mapping
emo_dict = common_dicts['emo_dict']
emo_color_dict = common_dicts['emo_color_dict']
usage_dict = common_dicts['usage_dict']

In [10]:
# Load model params from JSON
flat_models = load_config('./configs/flat_models.json')
print(flat_models.keys())

dict_keys(['LogisticRegression', 'DecisionTree', 'RandomForest', 'GradientBoosting', 'XGBoost', 'LGBM'])


In [11]:
# Modify df for clarity
df = df.rename(columns={'emotion': 'emotion_id'})
df['emotion'] = df['emotion_id'].astype(str).map(emo_dict)
df['color'] = df['emotion'].map(emo_color_dict)

In [12]:
# Re-map train-test split data
df['train_test_split'] = df['Usage'].map(usage_dict)

In [13]:
# ### Notebook currently runs flat array models, so no need for matrix representation yet
# Pixel data is read in as str, converted here to np.array 48x48
# df['image'] = df['pixels'].apply(convert_pixels_to_array)

In [14]:
# Pixel data is read in as str, converted here to a flat array (2304,)
df['flat_array'] = df['pixels'].apply(str_to_array)

In [15]:
df.columns

Index(['emotion_id', 'pixels', 'Usage', 'emotion', 'color', 'train_test_split',
       'flat_array'],
      dtype='object')

In [16]:
df.Usage.unique()

array(['Training', 'PublicTest', 'PrivateTest'], dtype=object)

In [17]:
df.train_test_split.unique()

array(['Train', 'Test'], dtype=object)

In [18]:
train_df = df[df['train_test_split']=='Train']
X_train = np.stack(train_df['flat_array'].values)
y_train = train_df['emotion']

In [19]:
# Verify splits
print(df.shape)
print(df.train_test_split.value_counts())
print(X_train.shape)
print(y_train.shape)

(35887, 7)
train_test_split
Train    28709
Test      7178
Name: count, dtype: int64
(28709, 2304)
(28709,)


In [20]:
# Create output dir
flat_model_dir = os.path.join('models', 'flat')
os.makedirs(flat_model_dir, exist_ok=True)

In [21]:
# Set paths for split data and settings
X_train_path = os.path.join(flat_model_dir, 'train_df.csv')
y_train_path = os.path.join(flat_model_dir, 'y_train.csv')
model_config_path = os.path.join(flat_model_dir, 'model_config.json')

In [22]:
# Save copy of split data and settings to model directory
train_df.to_csv(X_train_path, index=False)
y_train.to_csv(y_train_path, index=False)
to_json(flat_models, model_config_path)

## Flat Array Models

Trains each model specified in './configs/flat_models.json'

See flat_models dictionary for more details.

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

In [25]:
flat_models

{'LogisticRegression': {'module': 'sklearn.linear_model',
  'class': 'LogisticRegression',
  'params': {'multi_class': 'multinomial', 'max_iter': 500}},
 'DecisionTree': {'module': 'sklearn.tree',
  'class': 'DecisionTreeClassifier',
  'params': {}},
 'RandomForest': {'module': 'sklearn.ensemble',
  'class': 'RandomForestClassifier',
  'params': {}},
 'GradientBoosting': {'module': 'sklearn.ensemble',
  'class': 'GradientBoostingClassifier',
  'params': {}},
 'XGBoost': {'module': 'xgboost',
  'class': 'XGBClassifier',
  'params': {'eval_metric': 'mlogloss'}},
 'LGBM': {'module': 'lightgbm', 'class': 'LGBMClassifier', 'params': {}}}

In [26]:
models = create_models(flat_models)

In [27]:
flat_models

{'LogisticRegression': {'module': 'sklearn.linear_model',
  'class': 'LogisticRegression',
  'params': {'multi_class': 'multinomial', 'max_iter': 500}},
 'DecisionTree': {'module': 'sklearn.tree',
  'class': 'DecisionTreeClassifier',
  'params': {}},
 'RandomForest': {'module': 'sklearn.ensemble',
  'class': 'RandomForestClassifier',
  'params': {}},
 'GradientBoosting': {'module': 'sklearn.ensemble',
  'class': 'GradientBoostingClassifier',
  'params': {}},
 'XGBoost': {'module': 'xgboost',
  'class': 'XGBClassifier',
  'params': {'eval_metric': 'mlogloss'}},
 'LGBM': {'module': 'lightgbm', 'class': 'LGBMClassifier', 'params': {}}}

In [28]:
model_metrics = []

In [ ]:
for label, model in models.items():
    print(f"Running {label} model...")    
    # Set dirs and filepaths
    model_output_dir = os.path.join(flat_model_dir, label)
    model_output_path = os.path.join(model_output_dir, 'mdl.pkl')
    metrics_ouput_path = os.path.join(model_output_dir, 'train_metrics.csv')
    cm_ouput_path = os.path.join(model_output_dir, 'train_confusion_matrix.png')

    os.makedirs(model_output_dir, exist_ok=True)
    
    # fit, save, predict
    model.fit(X_train, y_train)
    save_model(model, filename=model_output_path)
    model_preds = model.predict(X_train)
    model_results = get_classification_metrics(y_train, model_preds)

    # Aggregate metrics and save to model dir
    pd.DataFrame(model_results, index=[0]).to_csv(metrics_ouput_path)
    model_metrics.append({label: model_results})

    # Confusion matrix
    cm_disp = ConfusionMatrixDisplay.from_predictions(
        y_true=y_train,
        y_pred=model_preds, 
        cmap='Blues'
    )    
    plt.tight_layout()
    plt.savefig(cm_ouput_path, pad_inches=5)

Running LogisticRegression model...


/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Running DecisionTree model...
Running RandomForest model...
Running GradientBoosting model...


In [ ]:
# Convert all model metrics to a df
agg_df = pd.DataFrame(model_metrics)

In [ ]:
# Save df
agg_df_path = os.path.join(flat_model_dir, 'aggregated_metrics.csv')
agg_df.to_csv(agg_df_path)